# 02 - Geometry: the emotion circumplex replicates (and what tuning does to it)

**What this notebook is for.** The first research question: do emotion vectors extracted from
Gemma-4-31B reproduce the published geometric structure of emotion space? Short answer: yes on
the base model, at the top of the reported range; instruction tuning then demotes the structure
without destroying it.

**Key concepts.**
- *Emotion vector*: the mean of the model's residual-stream activations while reading one
  emotion's stories, centered against the average across all 171 emotions.
- *Circumplex*: the psychology finding that emotions organize on two axes, valence
  (pleasant-unpleasant) and arousal (intensity). The papers report these as the top two
  directions of variation, found with principal component analysis (PCA).
- *NRC VAD*: the National Research Council valence-arousal-dominance lexicon, our human
  reference ratings (the paper used Russell's ratings; documented difference).
- *RSA*: representational similarity analysis, comparing whole similarity structures.

**Index.**
1. Valence tracking vs depth (the replication headline)
2. The circumplex at the peak layer
3. Similarity structure: synonym clusters
4. Cluster map (paper Figure 6)
5. Component loadings (paper Figure 7)
6. Stability across layers (RSA)
7. What instruction tuning changes

## 1. Valence tracking vs depth

In [1]:
# this cell loads the vector bundle, correlation results, and the NRC VAD lexicon
import json
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from scipy.cluster import hierarchy
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

ROOT = Path("..")  # notebooks/ lives one level below the repo root

bundle = np.load(ROOT / "results/emotion_vectors/emotion_means.npz", allow_pickle=True)
emotions, layers, means = list(bundle["emotions"]), list(bundle["layers"]), bundle["means"]
corr = json.loads((ROOT / "results/emotion_geometry_correlations.json").read_text())
vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
matched = [i for i, e in enumerate(emotions) if e.lower() in vad]
valence = np.array([vad[emotions[i].lower()][0] for i in matched])
arousal = np.array([vad[emotions[i].lower()][1] for i in matched])
print(f"{means.shape=}, layers {layers[0]}..{layers[-1]}, matched {len(matched)}/{len(emotions)}")

means.shape=(171, 20, 5376), layers 0..57, matched 164/171


In [2]:
# this cell plots |r| vs layer for PC1-valence and PC2-arousal, with published reference lines
per_layer = corr["per_layer"]
ls = [int(k) for k in per_layer]
r_val = [abs(per_layer[str(l)]["pc1_valence"]["pearson_r"]) for l in ls]
r_aro = [abs(per_layer[str(l)]["pc2_arousal"]["pearson_r"]) for l in ls]

fig = go.Figure()
fig.add_scatter(
    x=ls, y=r_val, mode="lines+markers", name="|r| PC1 vs NRC valence", line=dict(color="#1f77b4")
)
fig.add_scatter(
    x=ls,
    y=r_aro,
    mode="lines+markers",
    name="|r| PC2 vs NRC arousal",
    line=dict(color="#ff7f0e"),
    marker_symbol="square",
)
fig.add_hline(
    y=0.81,
    line_dash="dash",
    line_color="#1f77b4",
    opacity=0.6,
    annotation_text="Anthropic valence 0.81 (Russell)",
    annotation_position="bottom left",
)
fig.add_hline(
    y=0.83,
    line_dash="dot",
    line_color="#1f77b4",
    opacity=0.6,
    annotation_text="open replication 0.83 (NRC)",
    annotation_position="top left",
)
fig.add_hline(
    y=0.66,
    line_dash="dash",
    line_color="#ff7f0e",
    opacity=0.6,
    annotation_text="Anthropic arousal 0.66 (Russell)",
    annotation_position="bottom left",
)
fig.update_layout(
    title="Circumplex correlations by layer: Gemma-4-31B emotion vectors",
    xaxis_title="layer",
    yaxis_title="|Pearson r|",
    yaxis_range=[0, 1],
    height=450,
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Each point is one layer; height is how strongly that layer's principal component aligns with human ratings (absolute Pearson r, 1.0 is perfect). Dashed lines mark the published values; our curve reaching that band is the replication. Blue above orange everywhere means valence dominates arousal, as reported.

</details>

## 2. The circumplex at the peak layer

In [3]:
# this cell projects layer-33 vectors onto PC1/PC2 and draws the circumplex scatter
PEAK = layers.index(33)
scores = PCA(n_components=2).fit_transform(means[:, PEAK, :])
pc1, pc2 = scores[matched, 0], scores[matched, 1]
if np.corrcoef(pc1, valence)[0, 1] < 0:
    pc1, scores[:, 0] = -pc1, -scores[:, 0]
if np.corrcoef(pc2, arousal)[0, 1] < 0:
    pc2, scores[:, 1] = -pc2, -scores[:, 1]

sizes = 8 + 22 * (arousal - arousal.min()) / np.ptp(arousal)
fig = go.Figure(
    go.Scatter(
        x=scores[matched, 0],
        y=scores[matched, 1],
        mode="markers",
        marker=dict(
            color=valence,
            colorscale="RdYlGn",
            size=sizes,
            opacity=0.85,
            line=dict(color="black", width=0.3),
            colorbar=dict(title="NRC valence"),
        ),
        text=[str(emotions[i]) for i in matched],
        hoverinfo="text",
    )
)
extremes = np.argsort(np.abs(pc1))[-14:].tolist() + np.argsort(np.abs(pc2))[-10:].tolist()
for j in set(extremes):
    fig.add_annotation(
        x=float(pc1[j]),
        y=float(pc2[j]),
        text=str(emotions[matched[j]]),
        showarrow=False,
        xshift=6,
        yshift=9,
        font=dict(size=9),
        opacity=0.9,
    )
fig.update_layout(
    title="Emotion circumplex, layer 33 (point size = NRC arousal)",
    xaxis_title=f"PC1 (r={np.corrcoef(pc1, valence)[0, 1]:.3f} vs valence)",
    yaxis_title=f"PC2 (r={np.corrcoef(pc2, arousal)[0, 1]:.3f} vs arousal)",
    height=680,
    width=850,
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Each dot is one emotion projected on the top two principal components. A left-right color gradient means the first component encodes valence; larger dots (higher arousal) drifting upward means the second encodes arousal. A shuffled cloud would mean no circumplex.

</details>

## 3. Similarity structure: synonym clusters

In [4]:
# this cell clusters the 171x171 centered-cosine matrix and draws the heatmap
def centered_cosine(layer_means):
    c = layer_means - layer_means.mean(axis=0, keepdims=True)
    n = c / np.linalg.norm(c, axis=1, keepdims=True)
    return n @ n.T


sim = centered_cosine(means[:, PEAK, :])
order = hierarchy.leaves_list(hierarchy.linkage(1 - sim, method="average"))

ticks = list(range(0, len(order), 5))
labels = [str(emotions[order[t]]) for t in ticks]
fig = go.Figure(
    go.Heatmap(
        z=sim[np.ix_(order, order)],
        colorscale="RdBu",
        reversescale=True,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="cosine similarity (centered vectors)"),
    )
)
fig.update_layout(
    title="Pairwise emotion-vector similarity, layer 33, clustered",
    xaxis=dict(
        tickvals=ticks,
        ticktext=labels,
        tickangle=90,
        tickfont=dict(size=7),
        title="emotion (clustered order, every 5th labeled)",
    ),
    yaxis=dict(
        tickvals=ticks,
        ticktext=labels,
        tickfont=dict(size=7),
        title="emotion (clustered order, every 5th labeled)",
        autorange="reversed",
    ),
    height=780,
    width=860,
)
fig.show()

block = [emotions[i] for i in order[:8]]
print("first cluster block:", block)

first cluster block: [np.str_('ecstatic'), np.str_('euphoric'), np.str_('joyful'), np.str_('elated'), np.str_('exuberant'), np.str_('vibrant'), np.str_('cheerful'), np.str_('thrilled')]


<details><summary><b>How to read this figure</b></summary>

Rows and columns are the 171 emotions, ordered so similar vectors are adjacent. Red blocks on the diagonal are families of near-synonyms; blue regions are opposed emotions (usually opposite valence). A structureless field would mean no shared geometry.

</details>

## 4. Cluster map (paper Figure 6, embedding substituted)

In [5]:
# this cell clusters base-model emotion vectors and embeds them for display
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

bundle = np.load(ROOT / "results/emotion_vectors/emotion_means.npz", allow_pickle=True)
emotions = list(map(str, bundle["emotions"]))
layers = list(bundle["layers"])
M = bundle["means"][:, layers.index(33), :].astype(np.float64)
M -= M.mean(axis=0)

km = KMeans(n_clusters=10, n_init=10, random_state=20260722).fit(M)
xy = TSNE(n_components=2, perplexity=20, random_state=20260722).fit_transform(M)

fig = go.Figure()
for c in range(10):
    idx = np.where(km.labels_ == c)[0]
    fig.add_scatter(
        x=xy[idx, 0],
        y=xy[idx, 1],
        mode="markers+text",
        text=[emotions[i] for i in idx],
        textposition="top center",
        textfont=dict(size=7),
        name=f"cluster {c}",
        marker=dict(size=7),
    )
fig.update_layout(
    title="Emotion clusters, base model layer 33 (k-means k=10, t-SNE embedding)",
    xaxis_title="t-SNE dim 1",
    yaxis_title="t-SNE dim 2",
    height=650,
    showlegend=False,
)
fig.show()

for c in range(10):
    members = [emotions[i] for i in np.where(km.labels_ == c)[0]][:8]
    print(f"cluster {c}: {members}")

cluster 0: ['blissful', 'enthusiastic', 'grateful', 'happy', 'hopeful', 'inspired', 'invigorated', 'optimistic']
cluster 1: ['anxious', 'awestruck', 'bewildered', 'desperate', 'disoriented', 'dispirited', 'distressed', 'disturbed']
cluster 2: ['at ease', 'calm', 'content', 'fulfilled', 'hope', 'loving', 'patient', 'peaceful']
cluster 3: ['disdainful', 'greedy', 'relieved', 'self-confident', 'stubborn', 'valiant']
cluster 4: ['afraid', 'bored', 'brooding', 'dependent', 'depressed', 'docile', 'droopy', 'empathetic']
cluster 5: ['alarmed', 'amazed', 'annoyed', 'aroused', 'ashamed', 'astonished', 'bitter', 'defiant']
cluster 6: ['alert', 'angry', 'contemptuous', 'dumbstruck', 'embarrassed', 'enraged', 'exasperated', 'furious']
cluster 7: ['ecstatic', 'elated', 'euphoric', 'playful']
cluster 8: ['amused', 'cheerful', 'delighted', 'energized', 'excited', 'exuberant', 'joyful', 'jubilant']
cluster 9: ['compassionate', 'envious', 'indifferent', 'infatuated', 'jealous', 'lazy', 'lonely', 'paran

<details><summary><b>How to read this figure</b></summary>

Each point is an emotion, colored by k-means cluster (k=10, as the paper). We embed with t-SNE instead of the paper's UMAP (dependency choice; clustering identical). The paper's qualitative result reproduces: a joy/hope family, a calm/content family, a grief family.

</details>

## 5. Component loadings (paper Figure 7)

In [6]:
# this cell computes PCA loadings and draws the two ordered bar panels
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

from emotion_vectors.analysis import load_nrc_vad

vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
matched = [i for i, e in enumerate(emotions) if e.lower() in vad]
valence = np.array([vad[emotions[i].lower()][0] for i in matched])
arousal = np.array([vad[emotions[i].lower()][1] for i in matched])

pca = PCA(n_components=2)
scores = pca.fit_transform(M)
if np.corrcoef(scores[matched, 0], valence)[0, 1] < 0:
    scores[:, 0] *= -1
if np.corrcoef(scores[matched, 1], arousal)[0, 1] < 0:
    scores[:, 1] *= -1

fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        f"first principal component ({pca.explained_variance_ratio_[0]:.0%} variance, tracks valence)",
        f"second principal component ({pca.explained_variance_ratio_[1]:.0%} variance, tracks arousal)",
    ),
)
for row, comp in ((1, 0), (2, 1)):
    order = np.argsort(scores[:, comp])
    labels = [emotions[i] if r % 8 == 0 else "" for r, i in enumerate(order)]
    fig.add_bar(
        x=list(range(len(order))),
        y=scores[order, comp],
        row=row,
        col=1,
        marker_color=scores[order, comp],
        marker_colorscale="RdYlGn",
        showlegend=False,
    )
    fig.update_xaxes(
        tickvals=list(range(len(order))),
        ticktext=labels,
        tickangle=60,
        tickfont=dict(size=7),
        row=row,
        col=1,
    )
fig.update_layout(
    title="Emotion loadings on the top two principal components (base, layer 33)", height=700
)
fig.show()

<details><summary><b>How to read this figure</b></summary>

Bars are each emotion's score on the first (top) and second (bottom) principal component, sorted; sparse labels as in the paper. Reading the top panel left to right should walk from despair to joy; the bottom from serene to agitated.

</details>

## 6. Stability across layers

In [7]:
# this cell correlates similarity structures across layers (RSA) and draws the matrix
iu = np.triu_indices(len(emotions), k=1)
flat = np.stack([centered_cosine(means[:, p, :])[iu] for p in range(len(layers))])
rsa = np.corrcoef(flat)

fig = go.Figure(
    go.Heatmap(
        z=rsa,
        colorscale="Viridis",
        zmin=0,
        zmax=1,
        colorbar=dict(title="correlation of similarity structures"),
    )
)
fig.update_layout(
    title="Representational similarity analysis across layers",
    xaxis=dict(
        tickvals=list(range(len(layers))),
        ticktext=[str(l) for l in layers],
        title="layer",
        tickfont=dict(size=10),
    ),
    yaxis=dict(
        tickvals=list(range(len(layers))),
        ticktext=[str(l) for l in layers],
        title="layer",
        tickfont=dict(size=10),
        autorange="reversed",
    ),
    height=620,
    width=720,
)
fig.show()

mid = [p for p, l in enumerate(layers) if 30 <= l <= 57]
print(
    f"mean RSA within layers 30-57: {rsa[np.ix_(mid, mid)][np.triu_indices(len(mid), k=1)].mean():.3f}"
)
print(f"mean RSA layer 0 vs layers 30-57: {rsa[0, mid].mean():.3f}")

mean RSA within layers 30-57: 0.939
mean RSA layer 0 vs layers 30-57: 0.524


<details><summary><b>How to read this figure</b></summary>

Cell (i, j) asks: do layers i and j agree on which emotions resemble which? Bright middle-to-late block means the geometry consolidates early and stays stable, licensing single-layer analyses.

</details>

## 7. What instruction tuning changes

The same pipeline on the instruct model (`gemma-4-31b-it`) breaks the headline curve: valence
correlation collapses from layer 9 onward, while the first component's variance share doubles.
The next cell shows why that is a demotion, not a destruction: valence moves to the third
component, and the base model's valence direction still reads out valence in the instruct
model's vectors (r near 0.79). The instruct model adds dominant non-affective structure on
top of a preserved circumplex; the confound-projection experiment in the probe notebook removes
some but not all of it.

In [8]:
# this cell checks whether projection moves valence back toward PC1 (prediction P1)
LAYERS = layers  # swept layers, from the loader cell above
from scipy.stats import pearsonr
from sklearn.decomposition import PCA

from emotion_vectors.analysis import project_out_neutral

neutral = np.load(ROOT / "results/e7_neutral_bundle.npz")["vectors"].astype(np.float64)
vad = load_nrc_vad(ROOT / "data/lexicons/NRC-VAD-Lexicon-v2.1/NRC-VAD-Lexicon-v2.1.txt")
it_all = np.load(ROOT / "results/emotion_vectors_it_means.npz", allow_pickle=True)
all_emotions, all_means = list(map(str, it_all["emotions"])), it_all["means"].astype(np.float64)
matched = [i for i, e in enumerate(all_emotions) if e.lower() in vad]
valence = np.array([vad[all_emotions[i].lower()][0] for i in matched])
LP = LAYERS.index(33)

for name, M171 in (
    ("unprojected", all_means[:, LP, :]),
    (
        "projected",
        project_out_neutral(all_means[:, LP, :], neutral[:, LP, :].astype(np.float64))[0],
    ),
):
    centered = M171 - M171.mean(axis=0)
    pca = PCA(n_components=10).fit(centered)
    scores = pca.transform(centered)
    rs = [abs(float(pearsonr(scores[matched, k], valence).statistic)) for k in range(10)]
    best_pc = int(np.argmax(rs)) + 1
    print(
        f"{name:12s}: valence best at PC{best_pc} (|r|={max(rs):.3f}); "
        f"per-PC |r| {[round(r, 2) for r in rs[:5]]}"
    )

unprojected : valence best at PC3 (|r|=0.762); per-PC |r| [0.09, 0.1, 0.76, 0.29, 0.07]
projected   : valence best at PC2 (|r|=0.734); per-PC |r| [0.35, 0.73, 0.03, 0.16, 0.04]


<details><summary><b>How to read this output</b></summary>

For each model, the absolute correlation between valence and each of the top ten principal components. Base: component 1 carries valence (0.83). Instruct, unprojected: component 3 (0.76). Instruct, projected: component 2 (0.73). The demotion shrinks under neutral projection but the top component stays non-affective.

</details>